In [11]:
# 1/4 라이브러리 및 설정
import pandas as pd
import glob
import os

FOLDER = r'C:\Users\seohg\OneDrive\바탕 화면\2026\seoul datalob contest\store_name_1525\data'
OUTPUT = r'C:\Users\seohg\OneDrive\바탕 화면\seoul_map'

# 카페 업종 키워드
CAFE_KEYWORDS = ['커피', '카페', '음료', '티']

# 브랜드 키워드
BRAND_KEYWORDS = [
    '투썸', '이디야', '빽다방', '공차', '할리스',
    '파스쿠찌', '카페베네', '엔제리너스', '탐앤탐', 
    '폴바셋', '컴포즈', '메가커피', '요거프레소'
]

print('설정 완료')
print(f'카페 키워드: {CAFE_KEYWORDS}')
print(f'브랜드 수: {len(BRAND_KEYWORDS)}개')

설정 완료
카페 키워드: ['커피', '카페', '음료', '티']
브랜드 수: 13개


In [12]:
# 2/4 CSV 파일 목록 확인
files = sorted(glob.glob(f'{FOLDER}\\*.csv'))
print(f'총 파일 수: {len(files)}개')
print('첫 파일:', os.path.basename(files[0]))
print('마지막 파일:', os.path.basename(files[-1]))

총 파일 수: 40개
첫 파일: 201512.csv
마지막 파일: 202512.csv


In [13]:
# 3/4 파일별 브랜드 추출 (전체 병합 없이 처리)
results = []

cafe_pattern = '|'.join(CAFE_KEYWORDS)
brand_pattern = '|'.join(BRAND_KEYWORDS)

for file in files:
    period = os.path.basename(file).replace('.csv', '')[-6:]
    
    try:
        df = pd.read_csv(file, 
                         usecols=['상가업소번호', '상호명', '지점명',
                                  '상권업종소분류명', '시군구명', '행정동명',
                                  '건물관리번호', '층정보', '도로명주소',
                                  '경도', '위도'],
                         low_memory=False)
        
        # Step 1: 카페 업종만 필터링
        cafe_df = df[df['상권업종소분류명'].str.contains(cafe_pattern, na=False)]
        
        # Step 2: 브랜드 키워드로 필터링
        brand_df = cafe_df[cafe_df['상호명'].str.contains(brand_pattern, na=False)].copy()
        
        if len(brand_df) > 0:
            brand_df['period'] = f'df_{period}'
            results.append(brand_df)
        
        print(f'df_{period}: 카페 {len(cafe_df):,}개 → 브랜드 {len(brand_df):,}개')
        
    except Exception as e:
        print(f'df_{period} 오류: {e}')

print(f'\n처리 완료: {len(results)}개 파일')

df_201512: 카페 11,093개 → 브랜드 1,267개


KeyboardInterrupt: 

In [ ]:
# 4/4 병합 및 저장
brand_history = pd.concat(results, ignore_index=True)
print(f'총 브랜드 이력: {len(brand_history):,}행')
print()

# 브랜드별 집계
for kw in BRAND_KEYWORDS:
    cnt = brand_history['상호명'].str.contains(kw, na=False).sum()
    if cnt > 0:
        print(f'{kw}: {cnt:,}행')

# 저장
out_path = os.path.join(OUTPUT, 'brand_history.csv')
brand_history.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'\n✅ 저장 완료: {out_path}')

총 브랜드 이력: 74,948행

투썸: 10,433행
이디야: 23,783행
빽다방: 8,403행
공차: 5,090행
할리스: 4,178행
파스쿠찌: 2,501행
카페베네: 1,224행
엔제리너스: 1,984행
탐앤탐: 3,781행
폴바셋: 945행
컴포즈: 7,614행
메가커피: 1,459행
요거프레소: 3,553행

✅ 저장 완료: C:\Users\seohg\OneDrive\바탕 화면\seoul_map\brand_history.csv
